# LED OCR ablation, phase 2: extract pipeline inputs (Colab T4)
Protocol `led-ocr-ablation-261110-v1`. Run the SMOKE set first, read the timings, then run COHORT.
Videos and outputs stay on a private Drive folder, never in the repo. A failed video is a ledger row, not a rerun.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
# Edit these three paths once; keep them identical in the generate/score notebook.
REPO = '/content/multimodal-lecture-summarizer'                 # repo checkout (benchmark/ and multimodal-lecture-summarizer/)
WORK = '/content/drive/MyDrive/led-ocr-ablation-261110'         # private: videos/, run/
SET = 'smoke'                                                    # 'smoke' first, then 'cohort'
APP = f'{REPO}/multimodal-lecture-summarizer'
BENCH = f'{REPO}/benchmark'
PROTOCOL = f'{BENCH}/experiments/registry/protocols/led-ocr-ablation-261110-v1'
os.makedirs(f'{WORK}/videos', exist_ok=True); os.makedirs(f'{WORK}/run', exist_ok=True)

In [ ]:
# Frozen files must match their hashes before anything runs.
import hashlib, json
hashes = json.load(open(f'{PROTOCOL}/hashes.json'))['sha256']
for name, expected in hashes.items():
    assert hashlib.sha256(open(f'{PROTOCOL}/{name}', 'rb').read()).hexdigest() == expected, name
print('frozen files verified')

In [ ]:
!pip -q install -r {APP}/ai_workers/requirements.txt
!nvidia-smi -L

In [ ]:
# Resumable: rerun this cell after a disconnect. Videos with a ledger row are skipped.
!cd {BENCH} && python -m experiments.scripts.led_ocr_ablation_extract --app-root {APP} --videos-dir {WORK}/videos --out-dir {WORK}/run --set {SET} --protocol-dir {PROTOCOL}

In [ ]:
# Descriptive tables only: by stratum, OCR volume per video, segmenter fallbacks. Not used to drop videos.
import pandas as pd
rows = [json.loads(l) for l in open(f'{WORK}/run/extract_ledger_{SET}.jsonl')]
df = pd.DataFrame(rows)
cols = [c for c in ['id','stratum','status','segmenter','scenes','keyframes_with_ocr','raw_ocr_words','merge_ocr_words','merge_slides_used','merge_transcript_words','asr_model','wall_seconds','error'] if c in df]
display(df[cols])
if 'segmenter' in df: print('segmenter counts:', df['segmenter'].value_counts(dropna=False).to_dict())
if 'stratum' in df: display(df.groupby('stratum')[[c for c in ['merge_ocr_words','wall_seconds'] if c in df]].agg(['count','mean']))